# Självstudieövningar – Vecka 4

De här övningarna är till för att du ska repetera och befästa det vi har gått igenom under veckan samt introducera några nya begrepp på ett tryggt sätt.

Det innebär att vissa av uppgifterna kan du behöva söka efter information själv hur du löser. Här kan du använda Google eller AI om du vill men Python-dokumentationen är din bästa vän när du lär dig.

Länk: https://docs.python.org/3.13/

Läs först igenom vad som ska göras i respektive uppgift och fyll sedan i kodblocket nedanför, eller gör uppgiften i en separat fil.

**Detta lämnas inte in för rättning.** Arbeta i din egen takt, testa dig fram, använd dokumentationen när du kör fast och ta med eventuella frågor till nästa lektionstillfälle.

## (Kör först) Skapa veckans dataset

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # reproducible
n = 300
cities = np.array(["Göteborg","Malmö","Stockholm","Uppsala"])
genders = np.array(["F","M"])

age = rng.integers(18, 65, size=n)
gender = rng.choice(genders, size=n)
city = rng.choice(cities, size=n, p=[0.3,0.2,0.4,0.1])
steps = rng.normal(8000, 2000, size=n).clip(1000, 20000).astype(int)
sleep_hours = rng.normal(7.0, 1.2, size=n).clip(3.5, 11.0)
screen_time = rng.normal(4.0, 1.5, size=n).clip(0.5, 12.0)
calories = rng.normal(2200, 450, size=n).clip(1000, 4000)

# relationer: fler steg → lägre skärmtid, högre kalorier
screen_time = np.clip(screen_time - (steps - 8000)/8000 * 0.6, 0.5, 12.0)
calories    = np.clip(calories    + (steps - 8000)/8000 * 120, 1000, 4000)

df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "city": city,
    "steps": steps,
    "sleep_hours": np.round(sleep_hours,2),
    "screen_time": np.round(screen_time,2),
    "calories": np.round(calories,0).astype(int)
})

# ca 2% saknade värden (NaN)
mask = rng.random(size=df.shape) < 0.02
for (r, c), m in np.ndenumerate(mask):
    if m:
        df.iat[r, c] = np.nan

df.to_csv("week4_data.csv", index=False)
print("Skapade week4_data.csv med", len(df), "rader.")

### Om datasettet (week4_data.csv)

Det här är ett **syntetiskt** dataset från en fiktiv hälso-/livsstilsapp. Varje rad är en person/dag.

| Kolumn | Typ | Beskrivning | Ungefärligt intervall |
|---|---|---|---|
| `age` | int | Ålder | 18–64 |
| `gender` | kategori | Kön (F/M) | – |
| `city` | kategori | Stad | Göteborg, Malmö, Stockholm, Uppsala |
| `steps` | int | Antal steg/dygn | ~1 000–20 000 |
| `sleep_hours` | float | Sömn/dygn | ~3.5–11 |
| `screen_time` | float | Skärmtid/dygn | ~0.5–12 |
| `calories` | int | Energiförbrukning | ~1 000–4 000 |

**Noteringar:** Fast seed för reproducerbarhet, ~2% NaN, svag relation: fler `steps` → lägre `screen_time` men högre `calories` i snitt.

### Snabb kontroll
- Läs in `week4_data.csv` i `df`.
- Visa `df.head()` för att se de första raderna.
- Visa datatyper med `df.info()`.
- Räkna saknade värden per kolumn med en metod i Pandas.
- Skriv en kort kommentar (i en kodkommentar) om vilken kolumn som verkar ha flest NaN.

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

display(df.head())

df.info()

na_count = df.isna().sum()
print("\nSaknade värden per kolumn:")
print(na_count)

display(df.describe())

## Block A – NumPy

### Uppgift 1 – Array-grunder
Skapa en följd av heltal och gör enkla operationer med **vektoriserad** aritmetik.
- Skapa en 1D-array med heltal 0..19.
- Skapa en ny array med kvadraterna av dessa tal (utan loop).
- Plocka ut de tal som är delbara med 3 genom ett boolean-filter.

In [ ]:
import numpy as np

numbers = np.arange(20)
squares = numbers ** 2
divisible_by_three = numbers[numbers % 3 == 0]

print("Numbers:", numbers)
print("Squares:", squares)
print("Divisible by 3:", divisible_by_three)

### Uppgift 2 – Slicing & reshaping
Träna indexering i 2D.
- Skapa en array med tal 1..24.
- Forma om till en 4×6-matris.
- Plocka ut kolumnerna 2–4 (inklusive) och raderna 2–3 (inklusive) via slicing.

In [ ]:
import numpy as np

matrix = np.arange(1, 25).reshape(4, 6)
selected_part = matrix[1:3, 1:4]

print("Complete matrix:")
print(matrix)

print("\nRows 2–3 and columns 2–4:")
print(selected_part)

### Uppgift 3 – Broadcasting
Använd broadcasting för att kombinera en matris och en vektor.
- Skapa en 3×3-matris med ettor.
- Skapa en vektor `[10, 20, 30]`.
- Lägg till vektorn **radvis** till matrisen och skriv ut resultatet.

In [ ]:
import numpy as np

matrix = np.ones((3, 3))
vector = np.array([10, 20, 30])
result = matrix + vector

print(result)

### Uppgift 4 – Felsökning (listor vs arrayer)
Koden nedan försöker summera elementvis men använder listor. Kör och rätta.
```python
import numpy as np
a = [1,2,3]
b = [4,5,6]
print(a + b)      # blir listkonkatenering
print(a * b)      # TypeError
```
**Mål:** gör om till arrayer så att addition/multiplikation blir elementvis.

In [ ]:
import numpy as np
a = np.array([1,2,3])
b = np.array([4,5,6])
print(a + b)
print(a * b) 

### Uppgift 5 – Simulera och jämför tärningar

Simulera två omgångar med 1 000 tärningskast och jämför resultaten.

- Skapa först en slumptalsgenerator med `np.random.default_rng(42)`.
- Använd generatorns metod `integers()` för att skapa två separata simuleringar.
- Varje simulering ska innehålla 1 000 utfall mellan 1 och 6.
- Räkna hur många ettor, tvåor och så vidare som förekommer i respektive simulering.
- Jämför simuleringarna genom att beräkna den absoluta skillnaden mellan frekvenserna.
- Skriv ut resultatet som en liten tabell.  

*(tips: en funktion i NumPy: `np.bincount` kan räkna förekomster per heltal)*

In [ ]:
rng = np.random.default_rng(42)
s1 = rng.integers(1, 7, size=1000) #s = simulation
s2 = rng.integers(1, 7, size=1000)
f1 = np.bincount(s1 - 1, minlength=6) #f = frekvens (lite otydligt i videon)
f2 = np.bincount(s2 - 1, minlength=6)

diff = np.abs(f1 - f2)
pd.DataFrame({
    "utfall": np.arange(1,7),
    "sim1": f1,
    "sim2": f2,
    "abs_diff": diff
})

## Block B – Pandas (inkl. rensa data)

### Uppgift 6 – Skapa och inspektera DataFrame
- Skapa en DataFrame **manuellt** från en dictionary (3–4 rader), t.ex. namn/ålder/stad.
- Visa de första raderna.
- Visa datatyper och en numerisk sammanfattning.

In [ ]:
import pandas as pd

people = pd.DataFrame({
    "name": ["Joakim", "Hans", "Tommy", "Emil"],
    "age": [30, 25, 40, 22],
    "city": ["Göteborg", "Malmö", "Stockholm", "Uppsala"],
})

display(people.head())
people.info()
display(people.describe())

### Uppgift 7 – Välja & filtrera
- Läs `week4_data.csv`.
- Skapa en vy som bara innehåller kolumnerna `age`, `steps`, `city`.
- Filtrera fram rader där `steps` är större än 10 000 **och** `city` är Göteborg.
- Visa de 5 första raderna av resultatet.

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

selected_data = df[["age", "steps", "city"]]

gothenburg_over_10000 = selected_data[
    (selected_data["steps"] > 10000)
    & (selected_data["city"] == "Göteborg")
]

display(gothenburg_over_10000.head())

### Uppgift 8 – Sortera & skapa kolumner
- Sortera data fallande på `steps` (störst först).
- Skapa en ny kolumn `steps_k` som visar steg i tusental (avrunda till 1 decimal).
- Visa de 5 första raderna efter sorteringen.

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

sorted_data = df.sort_values(
    by="steps",
    ascending=False,
).copy()

sorted_data["steps_k"] = (sorted_data["steps"] / 1000).round(1)

display(sorted_data.head())

### Uppgift 9 – Saknade värden
- Räkna antalet saknade värden per kolumn.
- Fyll saknade i `sleep_hours` med **medianen**.
- Fyll saknade i `steps` med **medelvärdet**.
- Kontrollera igen att saknade värden minskat för dessa kolumner.

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

print("Missing values before:")
print(df.isna().sum())

median_sleep = df["sleep_hours"].median()
mean_steps = df["steps"].mean()

df["sleep_hours"] = df["sleep_hours"].fillna(median_sleep)
df["steps"] = df["steps"].fillna(mean_steps)

print(df[["sleep_hours", "steps"]].isna().sum())

### Uppgift 10 – Dubbletter

Skapa först en version av datasetet som innehåller några dubbletter:

```python
df_with_duplicates = pd.concat(
    [df, df.iloc[:3]],
    ignore_index=True
)
```
Arbeta därefter med df_with_duplicates:

- räkna hur många rader som är dubbletter,
- ta bort dubblettraderna,
- visa radantalet före och efter borttagningen,
- kontrollera att det rensade datasetet har samma antal rader som det ursprungliga df

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

df_with_duplicates = pd.concat(
    [df, df.iloc[:3]],
    ignore_index=True,
)

row_count_before = len(df_with_duplicates)
duplicate_count = df_with_duplicates.duplicated().sum()

df_without_duplicates = df_with_duplicates.drop_duplicates()
row_count_after = len(df_without_duplicates)

print("Dubletter:", duplicate_count)
print("Row count före:", row_count_before)
print("Row count efter:", row_count_after)
print("Samma row count som originalet:", row_count_after == len(df))

### Uppgift 11 – Typer och säker konvertering

Skapa först en kopia av datasetet där kolumnen `steps` lagras som text och ett av värdena är ogiltigt:

```python
df_dirty = df.copy()
df_dirty["steps"] = df_dirty["steps"].astype("string")
df_dirty.loc[0, "steps"] = "okänd"
```

Arbeta därefter med `df_dirty`:  
- kontrollera datatypen för `steps`,
- konvertera `steps` till numeriska värden på ett **säkert** sätt,
- se till att text som inte kan tolkas som ett tal blir `NaN`,
- fyll saknade värden i `steps` med kolumnens medelvärde,
- jämför datatyp och antal saknade värden före och efter konverteringen.

*(tips: titta på en Pandas-funktion som börjar `pd.to_num...`)*

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

df_dirty = df.copy()
df_dirty["steps"] = df_dirty["steps"].astype("string")
df_dirty.loc[0, "steps"] = "unknown"

print("Datatyp före:", df_dirty["steps"].dtype)
print("Saknade värden före konvertering:", df_dirty["steps"].isna().sum())

df_dirty["steps"] = pd.to_numeric(
    df_dirty["steps"],
    errors="coerce",
)

print("Datatyp efter konvertering:", df_dirty["steps"].dtype)
print(
    "Saknade värden efter konvertering:",
    df_dirty["steps"].isna().sum(),
)

mean_steps = df_dirty["steps"].mean()
df_dirty["steps"] = df_dirty["steps"].fillna(mean_steps)

print("Saknade värden efter ifyllnad:", df_dirty["steps"].isna().sum())

### Uppgift 12 – Rengöringspipeline
Gör en liten pipeline som kopplar ihop allt:
- Hantera saknade värden (se uppgift 9/11).
- Skapa en ny kolumn `active_ratio = steps / (screen_time * 1000)`.
- Filtrera bort orimliga rader (t.ex. `sleep_hours < 4` eller `> 12`).
- Gruppera per `city` och skriv ut medelvärden för `steps` och `active_ratio`.

In [ ]:
df["sleep_hours"] = df["sleep_hours"].fillna(df["sleep_hours"].median())
df["steps"] = pd.to_numeric(df["steps"], errors="coerce").fillna(df["steps"].mean())
df["screen_time"] = pd.to_numeric(df["screen_time"], errors="coerce")
df["active_ratio"] = df["steps"] / (df["screen_time"] * 1000)
clean = df[(df["sleep_hours"] >= 4) & (df["sleep_hours"] <= 12)]
res = (clean.groupby("city")
       .agg(steps_mean=("steps", "mean"),
            active_mean=("active_ratio", "mean"))
            .sort_values("steps_mean", ascending=False)
            )
display(res)

## Block C – NumPy + Pandas (deskriptiv statistik)

### Uppgift 13 – Centrala mått med NumPy
- Hämta kolumnen `steps` som en NumPy-array med `df["steps"].to_numpy()`.
- Ta bort eller ignorera saknade värden.
- Räkna ut medelvärde, median och stickprovsstandardavvikelse.
- Skriv ut resultaten med tydliga beskrivningar.

Använd `ddof=1` när du beräknar standardavvikelsen, eftersom observationerna här behandlas som ett stickprov.

*(tips: `np.mean`, `np.median`, `np.std`)*

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("week4_data.csv")

steps_array = df["steps"].to_numpy()
steps_without_missing = steps_array[~np.isnan(steps_array)]

mean_value = np.mean(steps_without_missing)
median = np.median(steps_without_missing)
sample_standard_deviation = np.std(steps_without_missing, ddof=1)

print(f"Medelvärde: {mean_value:.2f}")
print(f"Median: {median:.2f}")
print(
    "Stickprovsstandardavvikelse: "
    f"{sample_standard_deviation:.2f}"
)

### Uppgift 14 – Fem-tals-sammanfattning
För `sleep_hours`:
- Räkna fram min, Q1 (25%), median, Q3 (75%), max.
- Ignorera saknade värden först.  
*(tips: det finns en NumPy-funktion som heter `np.percentile`)*

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("week4_data.csv")

sleep_hours_array = df["sleep_hours"].to_numpy()
sleep_hours_without_missing = sleep_hours_array[~np.isnan(sleep_hours_array)]

minimum = np.min(sleep_hours_without_missing)
q1 = np.percentile(sleep_hours_without_missing, 25)
median = np.median(sleep_hours_without_missing)
q3 = np.percentile(sleep_hours_without_missing, 75)
maximum = np.max(sleep_hours_without_missing)

five_number_summary = {
    "minimum": minimum,
    "Q1": q1,
    "median": median,
    "Q3": q3,
    "maximum": maximum,
}

print(five_number_summary)

### Uppgift 15 – IQR och möjliga extremvärden

Undersök kolumnen `sleep_hours`:

- ta bort eller ignorera saknade värden,
- beräkna Q1 och Q3,
- beräkna `IQR = Q3 - Q1`,
- beräkna gränserna `low = Q1 - 1.5 * IQR` och `high = Q3 + 1.5 * IQR`,
- skapa en DataFrame `outliers_df` med rader som ligger utanför intervallet,
- skriv ut hur många möjliga extremvärden som hittades,
- visa kolumnen `sleep_hours` för dessa rader.

Värden utanför IQR-gränserna är möjliga extremvärden och behöver inte automatiskt vara felaktiga.

In [ ]:
df = pd.read_csv("week4_data.csv")

sleep_hours = df["sleep_hours"].dropna()
q1 = sleep_hours.quantile(0.25)
q3 = sleep_hours.quantile(0.75)

iqr = q3 - q1

low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

mask = (
    (df["sleep_hours"] < low)
    | (df["sleep_hours"] > high)
)

outliers_df = df[mask]
print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"Nedre gräns: {low}")
print(f"Övre gräns: {high}")
print(f"Antal möjliga extremvärden: {len(outliers_df)}")
display(outliers_df[["sleep_hours"]])

### Uppgift 16 – Felsökning (typkonvertering)
Koden försöker räkna medelvärde på blandade typer. Kör och rätta.
```python
import numpy as np
vals = ["3", "5", "7", None]
print(np.mean(vals))  # TypeError
```
**Mål:** gör om värdena till tal och ignorera ogiltiga poster.

In [ ]:
import numpy as np

values = ["3", "5", "7", None]
numeric_values = []

for value in values:
    try:
        numeric_values.append(float(value))
    except (TypeError, ValueError):
        pass

mean_value = np.mean(numeric_values)
print("Medelvärde:", mean_value)

### Uppgift 17 – Sammanfattning per stad

Skapa en gruppsammanställning med `city` som index och följande kolumner:

- `steps_mean` – medelvärdet av `steps`,
- `sleep_median` – medianen av `sleep_hours`,
- `screen_std` – standardavvikelsen för `screen_time`.

Sortera resultatet fallande efter `steps_mean`.

**Tips:** Pandas har så kallad *named aggregation*, där namn, ursprungskolumn och beräkning anges tillsammans:

```python
new_name=("original_column", "calculation")
```

In [ ]:
import pandas as pd

df = pd.read_csv("week4_data.csv")

summary_by_city = (
    df.groupby("city")
    .agg(
        steps_mean=("steps", "mean"),
        sleep_median=("sleep_hours", "median"),
        screen_std=("screen_time", "std"),
    )
    .sort_values(by="steps_mean", ascending=False)
)

display(summary_by_city)

## Block D – Matplotlib

### Uppgift 18 – Stapeldiagram från frekvenser
- Räkna frekvens av `city` med `value_counts()`.
- Rita ett **stapeldiagram**.
- Lägg till titel, axelrubriker och rutnät för att göra grafen lätt att läsa.
*(tips: du kan antingen använda `DataFrame.plot(...)` eller `matplotlib.pyplot`)*

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("week4_data.csv")

city_counts = df["city"].value_counts()

figure, axis = plt.subplots(figsize=(7, 4))
axis.bar(city_counts.index, city_counts.values)
axis.set_title("Antal observationer per stad")
axis.set_xlabel("Stad")
axis.set_ylabel("Antal")
axis.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Uppgift 19 – Fördelning (histogram)
- Rita ett **histogram** av `steps` (välj ett rimligt antal bins, t.ex. 20).
- Lägg till titel/axlar och ett rutnät.
- Skriv en kort kommentar i kod (t.ex. om fördelningen ser skev ut).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("week4_data.csv")
steps_without_missing = df["steps"].dropna()

figure, axis = plt.subplots(figsize=(7, 4))
axis.hist(steps_without_missing, bins=20, edgecolor="black")
axis.set_title("Fördelning av antal steg")
axis.set_xlabel("Steg per dygn")
axis.set_ylabel("Frekvens")
axis.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Uppgift 20 – Samband (scatter)
- Skapa en **scatterplot** med `steps` på x-axeln och `calories` på y-axeln.
- Lägg till titel, axelrubriker och rutnät.
- Skriv en kort kommentar om du tycker sambandet verkar positivt/negativt/svagt.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv("week4_data.csv")
data_for_diagram = df[["steps", "calories"]].dropna()

fig, ax = plt.subplots(figsize=(8,5))

ax.scatter(
    data_for_diagram["steps"],
    data_for_diagram["calories"],
    alpha=0.7
)

ax.set_title("Samband mellan antal steg och kalorier")
ax.set_xlabel("Antal steg")
ax.set_ylabel("Kalorier")
ax.grid(True, alpha=0.6)

### Uppgift 21 – Felsökning (plot)
Koden försöker rita ett histogram på en icke-existerande kolumn. Kör och rätta.
```python
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("week4_data.csv")
df["step"].plot(kind="hist", bins=10)
plt.show()
```
**Mål:** använd korrekt kolumn och visa grafen.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("week4_data.csv")

df["steps"].dropna().plot(kind="hist", bins=10)
plt.title("Histogram över antal steg")
plt.xlabel("Steg per dygn")
plt.ylabel("Frekvens")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Uppgift 22 – Mini-dashboard
Skapa tre figurer:
1. Histogram av `sleep_hours`,
2. Stapeldiagram med frekvenser för `gender`,
3. Scatter `steps` vs `screen_time`.

Lägg på titlar/axlar/rutnät och spara **en** figur till fil (valfritt filnamn).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("week4_data.csv")

figure_1, axis_1 = plt.subplots(figsize=(7, 4))
axis_1.hist(
    df["sleep_hours"].dropna(),
    bins=20,
    edgecolor="black",
)
axis_1.set_title("Fördelning av sömntimmar")
axis_1.set_xlabel("Sömn per dygn i timmar")
axis_1.set_ylabel("Frekvens")
axis_1.grid(axis="y", alpha=0.3)
figure_1.tight_layout()
figure_1.savefig("sleep_histogram.png", dpi=150)
plt.show()

gender_counts = df["gender"].value_counts()

figure_2, axis_2 = plt.subplots(figsize=(6, 4))
axis_2.bar(gender_counts.index, gender_counts.values)
axis_2.set_title("Antal observationer per kön")
axis_2.set_xlabel("Kön")
axis_2.set_ylabel("Antal")
axis_2.grid(axis="y", alpha=0.3)
figure_2.tight_layout()
plt.show()

steps_and_screen_time = df[["steps", "screen_time"]].dropna()

figure_3, axis_3 = plt.subplots(figsize=(7, 4))
axis_3.scatter(
    steps_and_screen_time["steps"],
    steps_and_screen_time["screen_time"],
    alpha=0.6,
)
axis_3.set_title("Samband mellan steg och skärmtid")
axis_3.set_xlabel("Steg per dygn")
axis_3.set_ylabel("Skärmtid per dygn i timmar")
axis_3.grid(alpha=0.3)
figure_3.tight_layout()
plt.show()
